In [1]:
from pathlib import Path
import math
import pandas as pd
from PIL import Image

import torch
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms

In [13]:
def find_project_root(marker_folder="data"):
    """
    Tries to find the project root by walking upward until it finds /data.
    Works from scripts, notebooks, or different working directories.
    """
    current = Path.cwd().resolve()

    for path in [current] + list(current.parents):
        if (path / marker_folder).exists():
            return path

    raise FileNotFoundError(
        f"Could not find project root. No folder named '{marker_folder}' found upward from {current}"
    )

In [42]:
class CaptchaDataset(Dataset):
    def __init__(
        self,
        data_dir="data/processed/5Char_2000_CapGen_grayscale",
        csv_name="ground_truth_index.csv",
        charset="2346789ABCDEFGHJKLMNPQRTUVWXYabcdefghijkmnpqrtuvwxy",
        label_length=5,
        valid_extensions={".jpg", ".jpeg", ".png"},
        return_filenames=False,
        subset_fraction=1.0,
    ):
        self.project_root = find_project_root()
        self.data_dir = (self.project_root / data_dir).resolve()
        self.csv_path = self.data_dir / csv_name

        self.charset = charset
        self.label_length = label_length
        self.valid_extensions = {ext.lower() for ext in valid_extensions}
        self.return_filenames = return_filenames

        # Character mappings
        self.char_to_idx = {char: idx for idx, char in enumerate(self.charset)}
        self.idx_to_char = {idx: char for char, idx in self.char_to_idx.items()}

        # Tensor conversion + normalize to [0,1]
        self.transform = transforms.ToTensor()

        # Basic path checks
        if not self.data_dir.exists():
            raise FileNotFoundError(f"Dataset folder not found: {self.data_dir}")

        if not self.csv_path.exists():
            raise FileNotFoundError(f"CSV file not found: {self.csv_path}")

        # Read labels CSV
        self.labels_df = pd.read_csv(self.csv_path)

        required_columns = {"filename", "label"}
        if not required_columns.issubset(self.labels_df.columns):
            raise ValueError(
                f"CSV must contain columns {required_columns}, found {set(self.labels_df.columns)}"
            )

        # Optional subset for quick testing
        if not (0 < subset_fraction <= 1.0):
            raise ValueError("subset_fraction must be in (0, 1].")

        if subset_fraction < 1.0:
            n_subset = max(1, int(len(self.labels_df) * subset_fraction))
            self.labels_df = self.labels_df.iloc[:n_subset].reset_index(drop=True)

        # Validate rows
        self.samples = []
        for _, row in self.labels_df.iterrows():
            filename = str(row["filename"]).strip()
            label = str(row["label"]).strip()

            file_path = self.data_dir / filename
            file_ext = file_path.suffix.lower()

            if file_ext not in self.valid_extensions:
                continue

            if not file_path.exists():
                continue

            if len(label) != self.label_length:
                continue

            if any(char not in self.char_to_idx for char in label):
                continue

            self.samples.append((file_path, label))

        if len(self.samples) == 0:
            raise ValueError("No valid samples found after validation.")

    def __len__(self):
        return len(self.samples)

    def encode_label(self, label):
        return torch.tensor(
            [self.char_to_idx[char] for char in label],
            dtype=torch.long
        )

    def decode_label(self, indices):
        return "".join(self.idx_to_char[int(idx)] for idx in indices)

    def __getitem__(self, idx):
        file_path, label = self.samples[idx]

        image = Image.open(file_path).convert("L")   # force grayscale
        image = self.transform(image)                # shape: [1, H, W], scaled to [0,1]

        label_tensor = self.encode_label(label)

        if self.return_filenames:
            return image, label_tensor, file_path.name

        return image, label_tensor


def create_dataloaders(
    data_dir="data/processed/5Char_2000_CapGen_grayscale",
    csv_name="ground_truth_index.csv",
    batch_size=32,
    train_ratio=0.7,
    val_ratio=0.15,
    test_ratio=0.15,
    random_seed=42,
    training=True,
    shuffle_train=True,
    num_workers=0,
    pin_memory=False,
    drop_last=False,
    charset="2346789ABCDEFGHJKLMNPQRTUVWXYabcdefghijkmnpqrtuvwxy",
    subset_fraction=1.0,
    return_filenames=False,
):
    # Split check
    total_ratio = train_ratio + val_ratio + test_ratio
    if not math.isclose(total_ratio, 1.0, rel_tol=1e-6):
        raise ValueError("train_ratio + val_ratio + test_ratio must equal 1.0")

    dataset = CaptchaDataset(
        data_dir=data_dir,
        csv_name=csv_name,
        charset=charset,
        label_length=5,
        return_filenames=return_filenames,
        subset_fraction=subset_fraction,
    )

    total_size = len(dataset)
    train_size = int(total_size * train_ratio)
    val_size = int(total_size * val_ratio)
    test_size = total_size - train_size - val_size

    generator = torch.Generator().manual_seed(random_seed)

    train_dataset, val_dataset, test_dataset = random_split(
        dataset,
        [train_size, val_size, test_size],
        generator=generator
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=shuffle_train if training else False,
        num_workers=num_workers,
        pin_memory=pin_memory,
        drop_last=drop_last
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=pin_memory,
        drop_last=False
    )

    test_loader = DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=pin_memory,
        drop_last=False
    )

    return train_loader, val_loader, test_loader, dataset.char_to_idx, dataset.idx_to_char

# Test of Complete DataLoader Class

In [41]:
# Quick test of dataloader
train_loader, val_loader, test_loader, char_to_idx, idx_to_char = create_dataloaders(
    batch_size=64,
    subset_fraction=0.1,   # small test first
)

print("Number of classes:", len(char_to_idx))
print("Train batches:", len(train_loader))
print("Val batches:", len(val_loader))
print("Test batches:", len(test_loader))

batch = next(iter(train_loader))
images, labels = batch

print("Image batch shape:", images.shape)   # expected: [B, 1, H, W]
print("Label batch shape:", labels.shape)   # expected: [B, 5]
print("First label tensor:", labels[0])

Number of classes: 51
Train batches: 3
Val batches: 1
Test batches: 1
Image batch shape: torch.Size([64, 1, 64, 192])
Label batch shape: torch.Size([64, 5])
First label tensor: tensor([ 2, 24,  3, 34, 47])


In [44]:
#Visual Sanity check
batch = next(iter(train_loader))
images, labels = batch

print(images.min().item(), images.max().item())   # should be around 0 to 1
print(labels[0])                                  # tensor of length 5

decoded = "".join(idx_to_char[int(i)] for i in labels[0])
print(decoded)

0.0 1.0
tensor([41, 33, 49, 27, 35])
nexXg


# Detailed tests(was needed for debugging)

In [23]:
# Confirm project root & dataset path
cwd = Path.cwd().resolve()
print("Current working dir:", cwd)

for p in [cwd] + list(cwd.parents):
    if (p / "data").exists():
        print("Detected project root:", p)
        print("Dataset path:", p / "data/processed/5Char_2000_CapGen_grayscale")
        print("CSV path:", p / "data/processed/5Char_2000_CapGen_grayscale/ground_truth_index.csv")
        break

Current working dir: /Users/marc/Documents/VS-Code Projects/captcha-ai/notebooks
Detected project root: /Users/marc/Documents/VS-Code Projects/captcha-ai
Dataset path: /Users/marc/Documents/VS-Code Projects/captcha-ai/data/processed/5Char_2000_CapGen_grayscale
CSV path: /Users/marc/Documents/VS-Code Projects/captcha-ai/data/processed/5Char_2000_CapGen_grayscale/ground_truth_index.csv


In [ ]:
# Confrim dataset files
root = Path.cwd().resolve()
for p in [root] + list(root.parents):
    if (p / "data").exists():
        root = p
        break

data_dir = root / "data/processed/5Char_2000_CapGen_grayscale"
print("Data dir exists:", data_dir.exists())

if data_dir.exists():
    files = list(data_dir.iterdir())
    print("Number of files in folder:", len(files))
    print("First 10 files:", [f.name for f in files[:10]])

Data dir exists: True
Number of files in folder: 2001
First 10 files: ['001927.png', '001099.png', '000387.png', '000393.png', '001933.png', '001700.png', '000378.png', '001066.png', '001072.png', '001714.png']


In [25]:
#Inspecting the CSV

csv_path = Path("data/processed/5Char_2000_CapGen_grayscale/ground_truth_index.csv")
if not csv_path.exists():
    for p in [Path.cwd().resolve()] + list(Path.cwd().resolve().parents):
        alt = p / "data/processed/5Char_2000_CapGen_grayscale/ground_truth_index.csv"
        if alt.exists():
            csv_path = alt
            break

df = pd.read_csv(csv_path)
print(df.head())
print(df.columns.tolist())
print("Rows:", len(df))

     filename  label
0  000001.png  XFHkb
1  000002.png  k8RXb
2  000003.png  BVKDn
3  000004.png  7KYp9
4  000005.png  aQPEM
['filename', 'label']
Rows: 2000


In [26]:
#Check Filename mismatch
data_dir = csv_path.parent
df = pd.read_csv(csv_path)

print("CSV first filename:", df.iloc[0]["filename"])
print("CSV first label:", df.iloc[0]["label"])

sample_name = str(df.iloc[0]["filename"]).strip()
sample_path = data_dir / sample_name

print("Sample path exists:", sample_path.exists())
print("Sample path:", sample_path)

CSV first filename: 000001.png
CSV first label: XFHkb
Sample path exists: True
Sample path: /Users/marc/Documents/VS-Code Projects/captcha-ai/data/processed/5Char_2000_CapGen_grayscale/000001.png


In [27]:
#Check labels are valid for charset and length
charset = "2346789ABCDEFGHJKLMNPQRTUVWXYabcdefghijkmnpqrtuvwxy"
char_set = set(charset)

bad_labels = []
wrong_length = []

for _, row in df.iterrows():
    label = str(row["label"]).strip()
    if len(label) != 5:
        wrong_length.append(label)
    elif any(c not in char_set for c in label):
        bad_labels.append(label)

print("Wrong-length labels:", len(wrong_length))
print("Bad-label labels:", len(bad_labels))
print("Examples wrong length:", wrong_length[:5])
print("Examples bad charset:", bad_labels[:5])

Wrong-length labels: 0
Bad-label labels: 0
Examples wrong length: []
Examples bad charset: []


In [28]:
#Load one sample image and label manually

sample_name = str(df.iloc[0]["filename"]).strip()
sample_path = data_dir / sample_name

img = Image.open(sample_path).convert("L")
print("Image size:", img.size)
print("Image mode:", img.mode)

Image size: (192, 64)
Image mode: L
